# 실습 2. vLLM 모델 서빙: 성능 테스트
이 노트북은 vLLM을 **OpenAI 호환 API 서버로 서빙**하고, 아래 5가지를 측정
1) Transformers vs vLLM 속도 비교
2) 단일 요청 Latency
3) 동시 요청(배치 효과) 비교
4) Throughput(tokens/sec)
5) 서빙 설정 변경(안정성 vs 성능)
---

**런타임 유형 변경 → GPU (A100)**

## 0) 설치

In [ ]:
# vLLM 및 관련 라이브러리 설치/재설치
# -q: Quiet 모드. 설치 로그를 최소화하여 화면을 깔끔하게 함.
# uninstall -y: 사용자 확인(y/n) 없이 즉시 삭제.
# 이는 Colab 환경에 기본 설치된 패키지들과의 버전 충돌을 방지하기 위함입니다.
!pip -q uninstall -y tensorflow-decision-forests google-generativeai google-ai-generativelanguage tensorflow grpcio-status

# vLLM 설치
# -U: 이미 설치되어 있다면 최신 버전(또는 지정 버전)으로 업그레이드.
# protobuf 버전을 고정하는 이유는 vLLM과의 의존성 충돌을 막기 위함입니다.
!pip -q install -U "protobuf>=6.33.0,<7" vllm==0.14.0

# OpenAI, Transformers, Accelerate 설치
# openai: vLLM 서버에 API 요청을 보내기 위한 클라이언트 라이브러리
# transformers: 비교군(Hugging Face 방식) 실행을 위해 필요
# accelerate: 모델을 GPU에 효율적으로 로드하기 위한 보조 라이브러리
!pip install openai transformers accelerate -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.4/495.4 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.0/355.0 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 144.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 123.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0

In [ ]:
# !pip -q install -U vllm openai transformers accelerate

## 1) vLLM OpenAI API 서버 실행
아래 서버는 `http://127.0.0.1:8000/v1` 로 OpenAI 호환 API를 제공

- GPU 인스턴스에서 Qwen2.5-1.5B Instruct 모델을 vLLM 기반 OpenAI 호환 API 서버로 실행
- 서버는 백그라운드에서 계속 실행되며, 로그는 /tmp/vllm_server.log 파일에 저장

- 주요 설정:

    - FP16(half) 사용 → GPU 메모리 절약
    - GPU 메모리 최대 75% 사용
    - 최대 Context Length: 4096 tokens
    - 0.0.0.0:8000 으로 외부 접근 허용
    - nohup + & 로 터미널 종료 후에도 서버 유지
    - tail 명령으로 서버 기동 상태 확인

In [ ]:
# !nohup: 리눅스 명령어. 터미널 세션이 끊겨도 프로세스가 죽지 않고 백그라운드에서 계속 돌게 함.
# python -m vllm.entrypoints.openai.api_server: 파이썬 모듈 모드로 vLLM의 OpenAI 호환 API 서버를 실행.
# \
#   줄바꿈을 의미하며, 하나의 긴 명령어를 여러 줄에 보기 좋게 쓰기 위함.

!nohup python -m vllm.entrypoints.openai.api_server \
  --model Qwen/Qwen2.5-1.5B-Instruct \
  --dtype half \
  --gpu-memory-utilization 0.75 \
  --max-model-len 4096 \
  --host 0.0.0.0 \
  --port 8000 \
  > /tmp/vllm_server.log 2>&1 &

# 위 명령어의 상세 옵션 설명:
# --model: 사용할 모델 지정 (Hugging Face Hub ID)
# --dtype half: FP16(반정밀도) 사용. 메모리는 절반만 쓰고 속도는 높임.
# --gpu-memory-utilization 0.75: GPU 메모리의 75%를 미리 점유하여 KV Cache로 사용. (OOM 방지 및 배치 성능 확보)
# --max-model-len 4096: 입력+출력 토큰 길이의 최대 합.
# --host 0.0.0.0: 모든 네트워크 인터페이스에서 접근 허용 (Colab 외부 접속은 안 되지만 로컬 통신용).
# --port 8000: 서버 포트 번호.
# > /tmp/vllm_server.log: 표준 출력을 파일로 저장 (로그 확인용).
# 2>&1: 표준 에러(2)도 표준 출력(1)과 같은 곳으로 보냄.
# &: 명령어를 백그라운드에서 실행 (이 셀이 끝나도 서버는 계속 돔).

# 서버가 제대로 떴는지 로그의 끝부분 25줄을 확인
!tail -n 25 /tmp/vllm_server.log

## 2) 클라이언트 연결 및 공통 유틸
- 로컬에서 실행 중인 vLLM OpenAI 호환 서버(http://127.0.0.1:8000)에 연결하여 LLM을 1회 호출하고 응답 시간(latency), 사용 토큰 수, 생성된 텍스트를 측정하는 간단한 함수

- 주요 포인트:
    - OpenAI SDK를 사용하지만 실제로는 vLLM 서버 호출
    - base_url로 로컬 API 서버 지정
    - chat_once() 함수는:
        - 요청 1회 수행
        - 응답 시간 측정
        - 전체 토큰 수 추출
        - 모델 출력 텍스트 반환

→ throughput와 latency 실험 사용

In [ ]:
from openai import OpenAI # OpenAI 공식 SDK import. 실제 OpenAI가 아니라 vLLM 서버에 요청을 보낼 것임.
import time # 실행 시간(Latency) 측정을 위한 모듈.
import statistics # 수집된 데이터의 평균, 중앙값 계산을 위한 통계 모듈.
import concurrent.futures # 동시성(병렬) 요청 테스트를 위한 모듈.

# 클라이언트 객체 생성
# base_url='http://127.0.0.1:8000/v1': 요청을 보낼 주소를 로컬 vLLM 서버로 바꿔치기함.
# api_key='EMPTY': 로컬 vLLM은 기본적으로 키 검사를 안 하므로 아무 값이나 넣음.
client = OpenAI(
    base_url='http://127.0.0.1:8000/v1',
    api_key='EMPTY'
)

MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct' # 호출할 모델 이름 (서버 띄울 때 쓴 모델명과 같아야 함)

def chat_once(prompt: str, max_tokens=200, temperature=0.0):
    """OpenAI 호환 API로 1회 호출하고 latency + total_tokens + text를 반환하는 함수"""

    # 1. 시작 시간 기록
    start = time.time()

    # 2. API 호출 (동기 방식)
    # client.chat.completions.create: 서버에 POST 요청을 보내고 응답이 올 때까지 기다림 (Blocking)
    resp = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{'role': 'user', 'content': prompt}], # 대화 내역 포맷
        temperature=temperature, # 0.0: 무작위성 제거 (항상 같은 답이 나오도록 하여 성능 측정 일관성 유지)
        max_tokens=max_tokens, # 생성할 최대 토큰 수 제한
    )

    # 3. 경과 시간 계산 (현재 시간 - 시작 시간)
    elapsed = time.time() - start

    # 4. 결과 반환
    # elapsed: 응답까지 걸린 시간 (초)
    # resp.usage.total_tokens: 입력 토큰 + 출력 토큰 수 (Throughput 계산용)
    # resp.choices[0].message.content: 모델이 뱉은 텍스트 내용
    return elapsed, resp.usage.total_tokens, resp.choices[0].message.content

## ✅ 테스트 1) Transformers vs vLLM 속도 비교

In [ ]:
time.sleep(20) # 로딩 시간 필요

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

prompt = 'RAG가 뭔지 중학생도 이해하게 8줄로 설명해줘.'
max_new_tokens = 200 # 생성할 토큰 개수

# === (A) Transformers (Hugging Face) 방식 ===
# 기계적 과정: 모델 가중치를 파이썬 메모리로 로드 -> GPU로 이동 -> 파이썬 루프 돌면서 토큰 하나씩 생성

# 1. 토크나이저 로드 (텍스트 -> 숫자 변환기)
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

# 2. 모델 로드
# torch_dtype=torch.float16: 메모리 절약을 위해 16비트 부동소수점 사용
# device_map='auto': 가용한 GPU에 자동으로 모델을 올림
hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto'
)

# 3. 입력 데이터 전처리 (텍스트 -> 텐서)
inputs = tok(prompt, return_tensors='pt').to(hf_model.device)

# 4. 추론 및 시간 측정
t0 = time.time()
with torch.no_grad(): # 학습이 아니므로 그라디언트 계산 끔 (메모리/속도 이득)
    # generate 함수 내부에서 for 루프를 돌며 토큰을 하나씩 생성함 (상대적으로 느림)
    out_ids = hf_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
hf_elapsed = time.time() - t0

# === (B) vLLM (서빙) 방식 ===
# 기계적 과정: HTTP 요청 -> vLLM 엔진(C++ 최적화, PagedAttention) -> 고속 생성 -> 응답
vllm_elapsed, vllm_tokens, vllm_text = chat_once(prompt, max_tokens=max_new_tokens, temperature=0.0)

# === 결과 출력 및 비교 ===
print('=== Test 1: Transformers vs vLLM(server) ===')
print(f'Transformers elapsed: {hf_elapsed:.2f}s')
print(f'vLLM(server) elapsed: {vllm_elapsed:.2f}s')
# Speedup 계산: HF 시간 / vLLM 시간. (예: 2배 빠르면 2.0x)
print(f'Speedup(hf/vllm): {hf_elapsed / vllm_elapsed:.2f}x')
print('--- vLLM sample output (first 300 chars) ---')
print(vllm_text[:300])

# [해설]
# 보통 vLLM이 HF Transformers보다 훨씬 빠릅니다 (수 배 이상).
# 이유는 vLLM의 'PagedAttention' 메모리 관리 기술과 최적화된 커널 때문입니다.

=== Test 1: Transformers vs vLLM(server) ===
Transformers elapsed: 6.35s
vLLM(server) elapsed: 0.81s
Speedup(hf/vllm): 7.80x
--- vLLM sample output (first 300 chars) ---
RAG는 "Revisable and Aggregated Knowledge"의 약자로, AI 기반의 학습 모델입니다. 이 모델은 두 가지 중요한 특징을 가지고 있습니다:

1. **Revisability**: RAG는 특정 질문에 대한 답변을 수정하거나 업데이트할 수 있는 능력이 있습니다. 즉, 질문에 새로운 정보를 추가하거나 잘못된 정보를 수정하는 것이 가능합니다.

2. **Aggregation**: 모델은 여러 데이터 세트에서 얻은 정보를 하나의 통합된 결과로 합치고, 이를 사용하여 질문에 대한 정확한 답변을 제공합니다.

이러


## ✅ 테스트 2) 단일 요청 Latency (5회)
- **실시간 응답성**

In [ ]:
prompt = 'Latency와 Throughput 차이를 5줄로 설명해줘.'
times = []

# 5번 반복하여 Latency(지연 시간) 측정
for _ in range(5):
    # chat_once 함수를 호출하여 API 응답 시간을 측정
    # _ : 반환값 중 텍스트 내용은 필요 없으므로 버림 (변수명 _는 관례적으로 '사용 안 함' 의미)
    elapsed, toks, _ = chat_once(prompt, max_tokens=160, temperature=0.0)
    times.append(elapsed) # 측정된 시간을 리스트에 저장

print('=== Test 2: Single-request latency (5회) ===')
# p50 (Median): 중앙값. 이상치(튀는 값)의 영향을 덜 받으므로 대표값으로 많이 씀.
print(f'p50: {statistics.median(times):.2f}s')
# min/max/mean: 최소, 최대, 평균 시간.
print(f'min: {min(times):.2f}s, max: {max(times):.2f}s, mean: {statistics.mean(times):.2f}s')

# [해설]
# 이 테스트는 '사용자 한 명이 채팅을 쳤을 때 얼마나 빨리 답이 나오는가'를 측정합니다.
# vLLM은 초기 로딩(Prefill)과 생성(Decode) 과정이 매우 최적화되어 있어 Latency가 낮습니다.

=== Test 2: Single-request latency (5회) ===
p50: 0.70s
min: 0.70s, max: 0.72s, mean: 0.70s


## ✅ 테스트 3) 동시 요청(배치 효과) 비교
- vLLM의 핵심 장점인 **continuous batching**

In [ ]:
# 5개의 서로 다른 프롬프트 준비
prompts = [
    'LLM이 뭐야? 3줄',
    '벡터DB는 왜 써? 3줄',
    '프롬프트 엔지니어링 팁 3개',
    '온프레미스 서빙 장점 3개',
    'KV cache가 뭔지 3줄',
]

# 헬퍼 함수: 프롬프트 하나를 받아 API 요청을 보내는 역할
def req(p):
    return chat_once(p, max_tokens=120, temperature=0.0)

# 1. 순차 처리 (Sequential)
# 하나 요청하고 -> 답 오면 -> 다음 거 요청 (직렬)
t0 = time.time()
seq = [req(p) for p in prompts] # 리스트 컴프리헨션으로 순서대로 실행
seq_wall = time.time() - t0

# 2. 동시 처리 (Concurrent)
# 5개를 거의 동시에 와라락 요청 보냄 (병렬)
t1 = time.time()
# ThreadPoolExecutor: 스레드를 여러 개 만들어 동시에 작업을 수행하게 해주는 도구
# max_workers=len(prompts): 프롬프트 개수만큼 일꾼(워커)을 생성
with concurrent.futures.ThreadPoolExecutor(max_workers=len(prompts)) as ex:
    # ex.map: 리스트의 각 요소에 대해 req 함수를 별도 스레드에서 실행
    par = list(ex.map(req, prompts))
par_wall = time.time() - t1

print('=== Test 3: Sequential vs Concurrent ===')
print(f'Sequential wall time: {seq_wall:.2f}s') # 순차적으로 걸린 총 시간
print(f'Concurrent wall time: {par_wall:.2f}s') # 동시에 처리해서 가장 늦게 끝난 것까지의 시간
# 배수 계산: 순차 시간 / 동시 시간
print(f'Speedup(seq/par): {seq_wall / par_wall:.2f}x')

# [해설]
# vLLM의 진가는 여기서 발휘됩니다. 'Continuous Batching' 기술 덕분에,
# 요청이 동시에 들어오면 묶어서 한 번에 GPU 연산을 수행합니다.
# 그래서 1개 처리하는 시간이나 5개 처리하는 시간이나 큰 차이가 없습니다 (Speedup이 높게 나옴).

=== Test 3: Sequential vs Concurrent ===
Sequential wall time: 2.01s
Concurrent wall time: 0.51s
Speedup(seq/par): 3.93x


## ✅ 테스트 4) Throughput(tokens/sec)
- 서버 처리량(규모)을 판단하는 지표를 계산
※ OpenAI 응답의 `usage.total_tokens`로 **대략적인** tok/s를 계산

In [ ]:
# Throughput: 초당 처리한 토큰 수 (Tokens per Second)
# 시스템이 얼마나 많은 작업량을 소화할 수 있는지 보여주는 지표

# 순차 처리 시 총 토큰 수 합계
# seq 리스트의 각 요소는 (elapsed, toks, text) 튜플임 -> 여기서 toks(인덱스 1)만 꺼내서 다 더함
seq_total_tokens = sum(toks for _, toks, _ in seq)

# 동시 처리 시 총 토큰 수 합계
par_total_tokens = sum(toks for _, toks, _ in par)

print('=== Test 4: Throughput (approx tokens/sec) ===')
# 계산: 총 처리 토큰 / 걸린 시간
print(f'Sequential tok/s: {seq_total_tokens / seq_wall:.2f}')
print(f'Concurrent tok/s: {par_total_tokens / par_wall:.2f}')

# [해설]
# Concurrent tok/s가 Sequential보다 월등히 높아야 정상입니다.
# 이는 vLLM 서버가 동시에 여러 요청을 효율적으로 '구겨 넣어서' 처리했음을 의미합니다.

=== Test 4: Throughput (approx tokens/sec) ===
Sequential tok/s: 325.32
Concurrent tok/s: 1287.05


## ✅ 테스트 5) 설정 변경(안정 vs 성능) 비교
- 실무 운영의 핵심인 **안정성 ↔ 성능 트레이드오프**

아래 순서대로 진행해 테스트 수행:
1) 서버 종료
2) 안정 우선 설정으로 재시작
3) 테스트 2~4(Latency/동시/Throughput)를 다시 실행해 결과 비교

### 5-A) 서버 종료

In [ ]:
# 기존 서버 프로세스 종료
# pkill -f: 프로세스 이름에 해당 문자열이 포함된 프로세스를 찾아 강제 종료(kill)함
# || true: 만약 종료할 프로세스가 없어서 에러가 나더라도 무시하고 다음으로 넘어가라는 쉘 명령어 관습
!pkill -f "vllm.entrypoints.openai.api_server" || true
print('서버 종료 완료')

^C
서버 종료 완료


### 5-B) 안정 우선 설정으로 재시작 (메모리 여유 ↑)

In [ ]:
# [안정성 강화 설정으로 서버 재시작]
# =================================================================
# [이 셀의 핵심: '안전 마진(Safety Margin)' 확보하기]
# 1. 점유율 조정(0.65): GPU 메모리의 65%만 vLLM이 미리 찜하게 합니다.
#    - 남은 35%는 시스템 오버헤드와 갑작스러운 연산 스파이크를 위한 '버퍼'가 됩니다.
#    - 그릇을 꽉 채우지 않아야 넘치지(OOM) 않고 장시간 안정적으로 돌아갑니다.
# 2. 길이 조정(2048): 한 번에 처리할 단어 수를 줄여서, 필수적으로 들어가는
#    메모리 총량 자체를 다이어트 시킨 설정입니다.
# =================================================================

# --gpu-memory-utilization 0.65: 
# "GPU 메모리 10GB 중 6.5GB만 내가 전용으로 쓸게. 나머지는 너(시스템)가 편하게 써."
# -> 이렇게 여유를 둬야 서버가 갑자기 죽는 현상을 막을 수 있습니다.

!nohup python -m vllm.entrypoints.openai.api_server \
  --model Qwen/Qwen2.5-1.5B-Instruct \
  --dtype half \
  --gpu-memory-utilization 0.65 \
  --max-model-len 2048 \
  --host 0.0.0.0 --port 8000 \
  > /tmp/vllm_server.log 2>&1 &

# 로그의 마지막 25줄을 보여주어 서버가 잘 켜지고 있는지 확인합니다.
!tail -n 25 /tmp/vllm_server.log
print('서버 재시작 완료: 메모리 여유 공간을 35%로 늘려 안정성을 확보했습니다.')

서버 재시작 완료: 안정 우선 설정


### 5-C) 테스트 2~4 다시 실행해서 비교
아래 셀은 테스트 2~4를 한 번에 재실행

In [ ]:
time.sleep(20)

In [ ]:
# [성능 재측정 및 결과 분석 셀]
# =================================================================
# [이 셀의 핵심: 서버 설정 변경 후 '성능 지표' 다시 읽기]
#
# 1. 목적: 메모리 점유율을 0.75에서 0.65로 낮춘 '안전 설정' 상태에서 
#    실제 서빙 성능이 얼마나 나오는지 확인하고 이전 데이터와 비교합니다.
#
# 2. 성능 지표 이해:
#    - Latency(지연 시간): 한 명이 질문했을 때 답변을 받는 데 걸리는 속도 (반응성)
#    - Throughput(처리량): 서버가 1초당 뱉어내는 총 토큰(단어) 수 (서버의 전체 파워)
#    - Speedup: 서버가 '동시 처리'를 얼마나 효율적으로 해서 시간을 아꼈는지 보여주는 배수
# =================================================================

# --- Re-Test 2: Latency (단일 요청 지연 시간) ---
# 측정 목표: "서버가 얼마나 빨리 대답을 시작하는가?" (사용자 체감 속도)
prompt = 'Latency와 Throughput 차이를 5줄로 설명해줘.'
times2 = []
for _ in range(5):
    # chat_once: 1회 요청 시간을 측정하는 커스텀 함수
    elapsed, toks, _ = chat_once(prompt, max_tokens=160, temperature=0.0)
    times2.append(elapsed)

print('=== Re-Test 2: Single-request latency (5회 반복 측정) ===')
# p50(중앙값): 가장 일반적인 속도 / mean(평균): 전체 평균 / min~max: 속도가 튀는지 확인
# 결과가 0.7초대라면, 한 명의 사용자가 대답을 다 받기까지 1초가 안 걸린다는 아주 쾌적한 상태입니다.
print(f'p50: {statistics.median(times2):.2f}s | mean: {statistics.mean(times2):.2f}s | min~max: {min(times2):.2f}~{max(times2):.2f}s')

# --- Re-Test 3: Concurrency (동시 처리 효율성) ---
# 측정 목표: "따로따로 5번 할 때 vs 한꺼번에 5개 할 때, 누가 더 빠른가?"
prompts2 = [
    'LLM이 뭐야? 3줄', '벡터DB는 왜 써? 3줄', '프롬프트 엔지니어링 팁 3개',
    '온프레미스 서빙 장점 3개', 'KV cache가 뭔지 3줄',
]
def req2(p):
    return chat_once(p, max_tokens=120, temperature=0.0)

# [방식 1] 순차 실행 (하나 끝날 때까지 기다렸다가 다음 질문)
t0 = time.time()
seq2 = [req2(p) for p in prompts2]
seq2_wall = time.time() - t0

# [방식 2] 동시 실행 (vLLM의 배치 처리 능력을 테스트하기 위해 5개를 한꺼번에 투하)
t1 = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=len(prompts2)) as ex:
    par2 = list(ex.map(req2, prompts2))
par2_wall = time.time() - t1

print('\n=== Re-Test 3: Sequential vs Concurrent (순차 vs 동시) ===')
# [결과 분석] 
# - Sequential(1.95s): 하나씩 했더니 약 2초가 걸림
# - Concurrent(0.51s): 동시에 했더니 1개 처리하는 시간(0.5초대)과 거의 비슷하게 끝남!
# - Speedup(3.85x): 5배까지는 아니지만, 약 4배에 가까운 시간을 절약했다는 뜻입니다.
print(f'Sequential wall time: {seq2_wall:.2f}s | Concurrent wall time: {par2_wall:.2f}s | Speedup: {seq2_wall / par2_wall:.2f}x')

# --- Re-Test 4: Throughput (초당 토큰 처리량) ---
# 측정 목표: "서버가 1초당 몇 단어를 만들어낼 수 있는가?" (서버의 가성비)
seq2_total_tokens = sum(toks for _, toks, _ in seq2)
par2_total_tokens = sum(toks for _, toks, _ in par2)

print('\n=== Re-Test 4: Throughput (1초당 생성 단어 수) ===')
# [결과 분석]
# - Sequential(336 tok/s): 1초에 약 336단어 생성
# - Concurrent(1294 tok/s): 1초에 약 1294단어 생성 (폭발적인 성능 향상)
# 묶어서 한 번에 처리(Batching)할 때 서버의 진가가 드러나며, 가성비가 4배 가까이 좋아집니다.
print(f'Sequential tok/s: {seq2_total_tokens / seq2_wall:.2f}')
print(f'Concurrent tok/s: {par2_total_tokens / par2_wall:.2f}')

# =================================================================
# [최종 결론 요약]
# 1. 성능 데이터의 마법: 
#    5개를 한꺼번에 처리해도(Concurrent), 1개 처리할 때랑 걸리는 시간(Wall time)이 
#    거의 비슷하다는 것이 vLLM 기술의 핵심입니다.
# 2. 설정 변경의 영향: 
#    메모리 점유율을 0.65로 낮추어 여유 공간을 35%나 비워두었음에도 불구하고,
#    동시 처리 시 1,300 tok/s에 육박하는 강력한 성능을 보여줍니다.
# 3. 튜닝 포인트: 
#    안정성(OOM 방지)을 위해 자원을 조금 덜 쓰더라도, vLLM의 배치 처리 알고리즘이 
#    성능 하락을 최소화해주므로 실무에서는 '안정성 위주' 설정을 권장합니다.
# =================================================================

=== Re-Test 2: Single-request latency (5회) ===
p50: 0.68s | mean: 0.70s | min~max: 0.68~0.76s
=== Re-Test 3: Sequential vs Concurrent ===
Sequential wall time: 1.95s | Concurrent wall time: 0.51s | Speedup: 3.85x
=== Re-Test 4: Throughput (approx tokens/sec) ===
Sequential tok/s: 336.19
Concurrent tok/s: 1294.52
